# CP3 — Customer Support (TechQA)

Diagnosis-driven greedy stage-wise selection (same approach as GK/hotpotqa).

**Domain notes (from team discussion):**
- No single chunking strategy works — config-based switching (sliding-window default, captures conversational flow; CS queries are short)
- Hybrid retrieval + reranker = strongest balance
- BGE-large / E5-large (general MTEB) over domain-specific
- Context placement: test reverse ordering (Lost in the Middle)
- Generator: include llama-4-scout; smaller judge models preferred but verify vs 70B judge (GK showed 8b judge over-flags → halved RMSE on upgrade, so run judge comparison before committing)

**Standing rules:** SEED=42, N_SAMPLES=25 (iterate) / 200 (final), k=5, timeout=30.0, triple backup every 5–10 examples, judge prompt pasted manually (Appendix 7.4, Friel et al. 2024), keys clipped, docs to judge as raw nested `[[key, sentence], ...]`.

In [1]:
# Cell 1 — Installs
!pip -q install datasets sentence-transformers faiss-cpu rank_bm25 groq scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.5 MB/s eta 0:00:00


In [2]:
# Cell 2 — Config
import os, json, time, random, math
import numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED)

DOMAIN       = "techqa"          # Customer Support
#N_SAMPLES    = 25                # 200 for final locked runs
N_SAMPLES    = 200                #  final locked runs
K            = 5                 # retrieval depth
TIMEOUT      = 30.0
CKPT_EVERY   = 5                 # triple backup every 5-10 examples

DRIVE_DIR    = "/content/drive/MyDrive/RAGBench_Results/CustomerSupport"
LOCAL_DIR    = "/content/results_cs"
REPO         = "veenulearns-lab/RAGBench-Capstone-Batch26"
REPO_SUBDIR  = "customer_support/results"

os.makedirs(LOCAL_DIR, exist_ok=True)

GROQ_MODELS = {
    "8b":      "llama-3.1-8b-instant",
    "70b":     "llama-3.3-70b-versatile",
    "scout":   "meta-llama/llama-4-scout-17b-16e-instruct",  # DECOMMISSIONED 2026-07-17
    "qwen":    "qwen/qwen3-32b",                              # DECOMMISSIONED 2026-07-17
    "gpt-oss": "openai/gpt-oss-120b",
}

In [3]:
# Cell 3 — Secrets + Groq rotation (variable key count)
from google.colab import userdata
from groq import Groq

GROQ_KEYS = []
for i in range(1, 11):
    try:
        k = userdata.get(f"GROQ_API_KEY_{i}")
        if k: GROQ_KEYS.append(k)
    except Exception:
        break
assert len(GROQ_KEYS) >= 5, f"Expected >=5 Groq keys, got {len(GROQ_KEYS)}"
print(f"{len(GROQ_KEYS)} Groq keys loaded")

GITHUB_TOKEN = userdata.get("GH_PAT_NEW")
assert GITHUB_TOKEN, "GH_PAT_NEW missing from Colab Secrets"

_key_idx = 0
def groq_client():
    global _key_idx
    c = Groq(api_key=GROQ_KEYS[_key_idx % len(GROQ_KEYS)], timeout=TIMEOUT)
    _key_idx += 1
    return c

def groq_chat(model, prompt, max_retries=8, temperature=0.0):
    for attempt in range(max_retries):
        try:
            r = groq_client().chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
            )
            out = r.choices[0].message.content
            if out is None:
                raise RuntimeError(f"Model {model} returned None — check model id")
            return out
        except Exception as e:
            if "rate_limit_exceeded" in str(e) and "per day" in str(e):
                if attempt == max_retries - 1:
                    raise RuntimeError(f"All keys exhausted (TPD) for {model}")
                continue   # drained key -> rotate immediately, no sleep
            wait = 2 ** attempt
            print(f"[groq_chat] {type(e).__name__}: {e} -> retry in {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"groq_chat failed after {max_retries} retries for {model}")

6 Groq keys loaded


In [4]:
# Cell 4 — Triple-backup checkpointing: local -> Drive -> GitHub
from google.colab import drive as gdrive
gdrive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)

import base64, requests

def _github_push(fname, content_str, retries=3):
    url = f"https://api.github.com/repos/{REPO}/contents/{REPO_SUBDIR}/{fname}"
    hdr = {"Authorization": f"token {GITHUB_TOKEN}"}
    for attempt in range(retries):
        r = requests.get(url, headers=hdr, timeout=TIMEOUT)
        sha = r.json().get("sha") if r.status_code == 200 else None
        payload = {"message": f"ckpt {fname}",
                   "content": base64.b64encode(content_str.encode()).decode()}
        if sha: payload["sha"] = sha
        r = requests.put(url, headers=hdr, json=payload, timeout=TIMEOUT)
        if r.status_code in (200, 201):
            return True
        if r.status_code == 409:
            time.sleep(1); continue   # stale SHA -> refetch and retry
        print(f"[github] push failed {r.status_code}: {r.text[:200]}"); return False
    print(f"[github] gave up after {retries} retries (409)"); return False

_last_ckpt_count = {}

def checkpoint(exp_id, records):
    if _last_ckpt_count.get(exp_id) == len(records):
        return   # nothing new since last backup -> skip double-push
    _last_ckpt_count[exp_id] = len(records)
    fname = f"{exp_id}.json"
    s = json.dumps(records, indent=2)
    with open(os.path.join(LOCAL_DIR, fname), "w") as f:  # 1) local
        f.write(s)
    with open(os.path.join(DRIVE_DIR, fname), "w") as f:  # 2) Drive
        f.write(s)
    ok = _github_push(fname, s)                            # 3) GitHub
    print(f"[ckpt] {exp_id}: {len(records)} records backed up (local/Drive{'/GitHub' if ok else ' — GitHub FAILED'})")

def load_checkpoint(exp_id):
    p = os.path.join(LOCAL_DIR, f"{exp_id}.json")
    if not os.path.exists(p):
        p = os.path.join(DRIVE_DIR, f"{exp_id}.json")
    if os.path.exists(p):
        with open(p) as f:
            recs = json.load(f)
        print(f"[ckpt] resumed {exp_id} at {len(recs)} records")
        return recs
    return []

Mounted at /content/drive


In [5]:
# Cell 5 — Load RAGBench techqa
from datasets import load_dataset

ds = load_dataset("rungalileo/ragbench", DOMAIN, split="test")
ds = ds.shuffle(seed=SEED).select(range(min(N_SAMPLES, len(ds))))
print(len(ds), "examples")
print({k: type(v).__name__ for k, v in ds[0].items()})

README.md:   0%|          | 0.00/24.7k [00:00<?, ?B/s]

techqa/train-00000-of-00001.parquet:   0%|          | 0.00/22.5M [00:00<?, ?B/s]

techqa/validation-00000-of-00001.parquet:   0%|          | 0.00/5.40M [00:00<?, ?B/s]

techqa/test-00000-of-00001.parquet:   0%|          | 0.00/5.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1192 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/304 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/314 [00:00<?, ? examples/s]

200 examples
{'id': 'str', 'question': 'str', 'documents': 'list', 'response': 'str', 'generation_model_name': 'str', 'annotating_model_name': 'str', 'dataset_name': 'str', 'documents_sentences': 'list', 'response_sentences': 'list', 'sentence_support_information': 'list', 'unsupported_response_sentence_keys': 'list', 'adherence_score': 'bool', 'overall_supported_explanation': 'str', 'relevance_explanation': 'str', 'all_relevant_sentence_keys': 'list', 'all_utilized_sentence_keys': 'list', 'trulens_groundedness': 'float', 'trulens_context_relevance': 'float', 'ragas_faithfulness': 'float', 'ragas_context_relevance': 'float', 'gpt3_adherence': 'float', 'gpt3_context_relevance': 'float', 'gpt35_utilization': 'float', 'relevance_score': 'float', 'utilization_score': 'float', 'completeness_score': 'float'}


In [6]:
# Cell 6 — Chunking (config-based switching; no single strategy for CS)
def chunk_fixed(text, size=120, overlap=0):
    words = text.split()
    step = max(size - overlap, 1)
    return [" ".join(words[i:i+size]) for i in range(0, len(words), step) if words[i:i+size]]

def chunk_sliding(text, size=120, overlap=40):
    # default for CS: sliding window preserves conversational flow around short queries
    return chunk_fixed(text, size=size, overlap=overlap)

def chunk_semantic(text, embedder, sim_thresh=0.55):
    import re
    sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]
    if len(sents) <= 1:
        return sents or [text]
    embs = embedder.encode(sents, normalize_embeddings=True)
    chunks, cur = [], [sents[0]]
    for i in range(1, len(sents)):
        if float(np.dot(embs[i-1], embs[i])) >= sim_thresh:
            cur.append(sents[i])
        else:
            chunks.append(" ".join(cur)); cur = [sents[i]]
    chunks.append(" ".join(cur))
    return chunks

CHUNKERS = {
    "fixed":    lambda t, emb=None: chunk_fixed(t),
    "sliding":  lambda t, emb=None: chunk_sliding(t),                       # 120w/40
    "sliding_L": lambda t, emb=None: chunk_sliding(t, size=300, overlap=100),  # large
    "semantic": lambda t, emb=None: chunk_semantic(t, emb),
}

def build_chunks(example, strategy, embedder=None):
    chunks = []
    for doc in example["documents"]:
        chunks.extend(CHUNKERS[strategy](doc, embedder))
    return chunks

In [7]:
# Cell 7 — Embedders (general MTEB models > domain-specific, per Biomedical finding)
from sentence_transformers import SentenceTransformer

EMB_MODELS = {
    "minilm":    "sentence-transformers/all-MiniLM-L6-v2",   # baseline
    "bge-large": "BAAI/bge-large-en-v1.5",
    "e5-large":  "intfloat/e5-large-v2",
}
_emb_cache = {}
def get_embedder(name):
    if name not in _emb_cache:
        _emb_cache[name] = SentenceTransformer(EMB_MODELS[name])
    return _emb_cache[name]

def embed_texts(embedder, texts, is_query=False, emb_name=""):
    if emb_name == "e5-large":  # e5 requires prefixes
        texts = [("query: " if is_query else "passage: ") + t for t in texts]
    return embedder.encode(texts, normalize_embeddings=True, show_progress_bar=False)

In [8]:
# Cell 8 — Indexing + retrieval (dense / bm25 / hybrid RRF)
# NOTE: embedder is ALWAYS passed explicitly — GK bug: stale global embedder in
# hybrid_retrieve() caused dimension mismatch for bge-large / e5-large.
import faiss
from rank_bm25 import BM25Okapi

def build_indexes(chunks, embedder, emb_name):
    embs = embed_texts(embedder, chunks, is_query=False, emb_name=emb_name).astype("float32")
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)
    bm25 = BM25Okapi([c.lower().split() for c in chunks])
    return index, bm25

def dense_retrieve(query, chunks, index, embedder, emb_name, k=K):
    q = embed_texts(embedder, [query], is_query=True, emb_name=emb_name).astype("float32")
    _, idx = index.search(q, min(k, len(chunks)))
    return [int(i) for i in idx[0] if i >= 0]

def bm25_retrieve(query, chunks, bm25, k=K):
    scores = bm25.get_scores(query.lower().split())
    return list(np.argsort(scores)[::-1][:k])

def hybrid_retrieve(query, chunks, index, bm25, embedder, emb_name, k=K, rrf_k=60, pool=20):
    d = dense_retrieve(query, chunks, index, embedder, emb_name, k=pool)
    b = bm25_retrieve(query, chunks, bm25, k=pool)
    rrf = {}
    for rank, i in enumerate(d): rrf[i] = rrf.get(i, 0) + 1.0 / (rrf_k + rank + 1)
    for rank, i in enumerate(b): rrf[i] = rrf.get(i, 0) + 1.0 / (rrf_k + rank + 1)
    return [i for i, _ in sorted(rrf.items(), key=lambda x: -x[1])[:k]]

In [9]:
# Cell 9 — Cross-encoder reranker (hybrid + reranker = strongest balance for CS)
from sentence_transformers import CrossEncoder
_reranker = None
def get_reranker():
    global _reranker
    if _reranker is None:
        _reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    return _reranker

def rerank(query, chunks, cand_ids, k=K, pool_mult=2):
    # retrieve wider pool upstream (e.g. 2k), rerank down to k
    pairs = [(query, chunks[i]) for i in cand_ids]
    scores = get_reranker().predict(pairs)
    order = np.argsort(scores)[::-1][:k]
    return [cand_ids[i] for i in order]

In [10]:
# # Cell 10 — Context ordering + generator
# def order_context(ids, mode="normal"):
#     # "reverse": best chunk last (closest to question) — Lost in the Middle test
#     return list(reversed(ids)) if mode == "reverse" else list(ids)

# GEN_PROMPT = """Answer the question using ONLY the provided context. If the context does not contain the answer, say "I don't know".

# Context:
# {context}

# Question: {question}

# Answer:"""

# def generate(question, chunks, ids, gen_model, ordering="normal"):
#     ids = order_context(ids, ordering)
#     context = "\n\n".join(chunks[i] for i in ids)
#     return groq_chat(GROQ_MODELS[gen_model], GEN_PROMPT.format(context=context, question=question))

In [11]:
# Cell 10 — Context ordering + generator (prompt variants + optional compression) Prompt variation
def order_context(ids, mode="normal"):
    return list(reversed(ids)) if mode == "reverse" else list(ids)

GEN_PROMPTS = {
    "strict": """Answer the question using ONLY the provided context. If the context does not contain the answer, say "I don't know".

Context:
{context}

Question: {question}

Answer:""",

    "soft": """Answer the question using the provided context. If the context only partially covers the question, answer what you can from the context and clearly state what information is missing. Only say "I don't know" if the context contains nothing relevant.

Context:
{context}

Question: {question}

Answer:""",
}

COMPRESS_PROMPT = """Extract from the following context only the information relevant to answering the question. Keep exact error codes, version numbers, and technical identifiers verbatim. If nothing is relevant, say "NO RELEVANT CONTENT".

Question: {question}

Context:
{context}

Relevant information:"""

def generate(question, chunks, ids, gen_model, ordering="normal",
             prompt_variant="strict", compress=False):
    ids = order_context(ids, ordering)
    context = "\n\n".join(chunks[i] for i in ids)
    if compress:
        context = groq_chat(GROQ_MODELS[gen_model],
                            COMPRESS_PROMPT.format(question=question, context=context))
    return groq_chat(GROQ_MODELS[gen_model],
                     GEN_PROMPTS[prompt_variant].format(context=context, question=question))

In [12]:
# Cell 11 — JUDGE PROMPT — *** PASTE MANUALLY ***
# Paste the verbatim Appendix 7.4 judge prompt (Friel et al. 2024) below.
# Do NOT let anything auto-generate this. Must be used with .format() using
# EXACTLY: documents=, question=, answer=
# (If the pasted prompt contains literal JSON braces, escape them as {{ }} or
#  switch to the .replace() pattern used in CP3_Biomedical.)

JUDGE_PROMPT = """I asked someone to answer a question based on one or more
documents. Your task is to review their response and assess whether or not each
sentence in that response is supported by text in the documents. And if so, which
sentences in the documents provide that support. You will also tell me which
of the documents contain useful information for answering the question, and
which of the documents the answer was sourced from.

Here are the documents, each of which is split into sentences. Alongside each
sentence is associated key, such as '0a.' or '0b.' that you can use to refer
to it:

```
{documents}
```

The question was:
```
{question}
```

Here is their response, split into sentences. Alongside each sentence is
associated key, such as 'a.' or 'b.' that you can use to refer to it. Note
that these keys are unique to the response, and are not related to the keys
in the documents:

```
{answer}
```

You must respond with a JSON object matching this schema:

{{
  "relevance_explanation": string,
  "all_relevant_sentence_keys": [string],
  "overall_supported_explanation": string,
  "overall_supported": boolean,
  "sentence_support_information": [
    {{
      "response_sentence_key": string,
      "explanation": string,
      "supporting_sentence_keys": [string],
      "fully_supported": boolean
    }}
  ],
  "all_utilized_sentence_keys": [string]
}}

The relevance_explanation field is a string explaining which documents
contain useful information for answering the question. Walk through the
information in the documents step by step and how it is useful for
answering the question.

The all_relevant_sentence_keys field is a list of all document sentence
keys (e.g. '0a') that are relevant to the question. Include every sentence
that is useful and relevant to the question, even if it was not used in the
response, or if only parts of the sentence are useful. Base this judgement
only on the documents and the question -- ignore the response entirely when
deciding relevance. Leave out sentences that could be removed from the
document without affecting someone's ability to answer the question.

The overall_supported_explanation field is a string explaining why the
response *as a whole* is or is not supported by the documents. Walk through
each claim in the response individually and assess its support (or lack of
support) in the documents one at a time, before drawing any conclusion about
the response as a whole.

The overall_supported field is a boolean reflecting the conclusion you
reached at the end of overall_supported_explanation: whether the response as
a whole is supported by the documents.

The sentence_support_information field is a list of objects, one for each sentence
in the response. Each object MUST have the following fields:
- response_sentence_key: a string identifying the sentence in the response. This
key is the same as the one used in the response above.- explanation: a string
explaining why the sentence is or is not supported by the documents.
- supporting_sentence_keys: keys (e.g. ’0a’) of sentences from the documents that
support the response sentence. If the sentence is not supported, this list MUST
be empty. If the sentence is supported, this list MUST contain one or more keys.
In special cases where the sentence is supported, but not by any specific sentence,
you can use the string "supported_without_sentence" to indicate that the sentence
is generally supported by the documents. Consider cases where the sentence is
expressing inability to answer the question due to lack of relevant information
in the provided contex as "supported_without_sentence". In cases
where the sentence is making a general statement (e.g. outlining the steps to produce
an answer, or summarizing previously stated sentences, or a transition sentence), use
the sting "general".In cases where the sentence is correctly stating a well-known fact,
like a mathematical formula, use the string "well_known_fact". In cases where the
sentence is performing numerical reasoning (e.g. addition, multiplication), use
the string "numerical_reasoning".
- fully_supported: a boolean indicating whether the sentence is fully supported by
the documents.
  - This value should reflect the conclusion  you drew at the end of your step-by-step
    breakdown in explanation.
  - If supporting_sentence_keys is an empty list, then fully_supported must be false.
  - Otherwise, use fully_supported to clarify whether everything in the response
  sentence is fully supported by the document text indicated in supporting_sentence_keys
  (fully_supported = true), or whether the sentence is only partially or incompletely
  supported by that document text (fully_supported = false).

The all_utilized_sentence_keys field is a list of all sentences keys (e.g. ’0a’) that
were used to construct the answer. Include every sentence that either directly supported
the answer, or was implicitly used to construct the answer, even if it was not used
in its entirety. Omit sentences that were not used, and could have been removed from
the documents without affecting the answer.

You must respond with a valid JSON string. Use escapes for quotes, e.g. ‘\\"‘, and
newlines, e.g. ‘\\n‘. Do not write anything before or after the JSON string. Do not
wrap the JSON string in backticks like ‘‘‘ or ‘‘‘json.

As a reminder: your task is to review the response and assess which documents contain
useful information pertaining to the question, and how each sentence in the response
is supported by the text in the documents."""

assert "PASTE APPENDIX" not in JUDGE_PROMPT, "Judge prompt not pasted yet!"

In [13]:
# Cell 12 — Sentence keying + judge call (docs AND answer as RAW NESTED keyed lists)
import re, string

def sentence_split(text):
    # techqa docs are logs/technotes with little punctuation -> split on newlines too
    parts = re.split(r"(?<=[.!?])\s+|\n+", text)
    return [s.strip() for s in parts if s.strip()]

def key_documents(chunks, ids):
    # [["0a", sent], ["0b", sent], ...] — raw nested list, never prose
    keyed = []
    for d_i, cid in enumerate(ids):
        for s_i, sent in enumerate(sentence_split(chunks[cid])):
            keyed.append([f"{d_i}{string.ascii_lowercase[s_i % 26]}", sent])
    return keyed

def key_answer(answer):
    # Appendix 7.4 requires the response split + keyed ("a.", "b.", ...) too
    return [[string.ascii_lowercase[i % 26], s] for i, s in enumerate(sentence_split(answer))]

def judge(judge_model, keyed_docs, question, answer):
    prompt = JUDGE_PROMPT.format(
        documents=json.dumps(keyed_docs),      # raw nested [[key, sentence], ...]
        question=question,
        answer=json.dumps(key_answer(answer)), # keyed response, same nested format
    )
    raw = groq_chat(GROQ_MODELS[judge_model], prompt)
    m = re.search(r"\{.*\}", raw, re.DOTALL)
    if not m:
        raise ValueError(f"Judge returned no JSON: {raw[:200]}")
    return json.loads(m.group(0))

In [14]:
# Cell 13 — TRACe metrics (corrected formulas) + key clipping + 7.4 field names
def _j_get(d, *names):
    for n in names:
        if n in d and d[n] is not None:
            return d[n]
    return []

def compute_trace(judge_out, keyed_docs):
    real_keys = {k for k, _ in keyed_docs}

    # Appendix 7.4 output fields are all_-prefixed. CLIP hallucinated keys
    # (unclipped keys -> impossible RMSE > 1.0, cuad incident).
    rel_raw  = _j_get(judge_out, "all_relevant_sentence_keys", "relevant_sentence_keys")
    util_raw = _j_get(judge_out, "all_utilized_sentence_keys", "utilized_sentence_keys")
    s_rel  = {k for k in rel_raw  if k in real_keys}
    s_util = {k for k in util_raw if k in real_keys}
    s_all  = real_keys

    # corrected denominators (mentor-approved):
    relevance    = len(s_rel)  / len(s_all) if s_all else 0.0   # / all retrieved keys
    utilization  = len(s_util) / len(s_all) if s_all else 0.0   # / all retrieved keys
    completeness = len(s_rel & s_util) / len(s_rel) if s_rel else 0.0  # / relevant keys

    # adherence: overall_supported when present, else all fully_supported
    if "overall_supported" in judge_out and judge_out["overall_supported"] is not None:
        adherence = 1 if judge_out["overall_supported"] else 0
    else:
        supp = judge_out.get("sentence_support_information", [])
        adherence = 1 if supp and all(s.get("fully_supported", False) for s in supp) else 0

    return {"relevance": relevance, "utilization": utilization,
            "completeness": completeness, "adherence": adherence,
            "_dropped_keys": sorted((set(rel_raw) | set(util_raw)) - real_keys)}

In [15]:
# Cell 14 — Scoring vs ground truth: RMSE + AUCROC (fallback to accuracy)
from sklearn.metrics import roc_auc_score, mean_squared_error

def score_experiment(records):
    out = {}
    for m in ["relevance", "utilization", "completeness"]:
        pred = [r["pred"][m] for r in records]
        gt   = [r["gt"][m]   for r in records if r["gt"][m] is not None]
        pr   = [r["pred"][m] for r in records if r["gt"][m] is not None]
        out[f"{m}_rmse"] = math.sqrt(mean_squared_error(gt, pr)) if gt else None

    adh_gt   = [r["gt"]["adherence"]   for r in records if r["gt"]["adherence"] is not None]
    adh_pred = [r["pred"]["adherence"] for r in records if r["gt"]["adherence"] is not None]
    if len(set(adh_gt)) > 1:
        out["adherence_aucroc"] = roc_auc_score(adh_gt, adh_pred)
    else:
        out["adherence_aucroc"] = None   # single class (GK case) — use accuracy
    out["adherence_accuracy"] = (np.mean([p == g for p, g in zip(adh_pred, adh_gt)])
                                 if adh_gt else None)
    out["idk_rate"] = np.mean([r["answer"].strip().lower().startswith("i don")
                               for r in records if r.get("answer")])   # <- here
    return out

def gt_from_example(ex):
    return {
        "relevance":    ex.get("relevance_score"),
        "utilization":  ex.get("utilization_score"),
        "completeness": ex.get("completeness_score"),
        "adherence":    (int(ex["adherence_score"]) if ex.get("adherence_score") is not None else None),
    }
# Reminder: 25-sample AUCROC structurally unstable (Finance finding) — 200-sample is authoritative.

In [16]:
# Cell 15 — Pipeline runner with resume + triple-backup every CKPT_EVERY
def run_experiment(exp_id, cfg):
    """cfg keys: chunking, embedding, retrieval(dense|bm25|hybrid),
    rerank(bool), ordering(normal|reverse), generator, judge"""
    print(f"=== {exp_id} | {cfg}")
    records = load_checkpoint(exp_id)
    done = {r["idx"] for r in records}
    embedder = get_embedder(cfg["embedding"])
    k_cfg = cfg.get("k", K)

    for idx in range(len(ds)):
        if idx in done:
            continue
        ex = ds[idx]
        try:
            chunks = build_chunks(ex, cfg["chunking"], embedder)
            index, bm25 = build_indexes(chunks, embedder, cfg["embedding"])
            pool = k_cfg * 2 if cfg.get("rerank") else k_cfg
            if cfg["retrieval"] == "dense":
                ids = dense_retrieve(ex["question"], chunks, index, embedder, cfg["embedding"], k=pool)
            elif cfg["retrieval"] == "bm25":
                ids = bm25_retrieve(ex["question"], chunks, bm25, k=pool)
            else:
                ids = hybrid_retrieve(ex["question"], chunks, index, bm25, embedder, cfg["embedding"], k=pool)
            if cfg.get("rerank"):
                ids = rerank(ex["question"], chunks, ids, k=k_cfg)

            # answer = generate(ex["question"], chunks, ids, cfg["generator"], cfg.get("ordering", "normal"))
            # keyed = key_documents(chunks, ids)
            # j = judge(cfg["judge"], keyed, ex["question"], answer)
            # pred = compute_trace(j, keyed)

            answer = generate(ex["question"], chunks, ids, cfg["generator"],
                              cfg.get("ordering", "normal"),
                              prompt_variant=cfg.get("prompt", "strict"),
                              compress=cfg.get("compress", False))
            keyed = key_documents(chunks, ids)
            j = judge(cfg["judge"], keyed, ex["question"], answer)
            pred = compute_trace(j, keyed)

            dropped = pred.pop("_dropped_keys", [])
            if dropped:
                print(f"[{exp_id}] idx {idx}: clipped {len(dropped)} hallucinated keys {dropped[:5]}")
            records.append({"idx": idx, "question": ex["question"], "answer": answer,
                            "pred": pred, "gt": gt_from_example(ex)})


            # dropped = pred.pop("_dropped_keys", [])
            # if dropped:
            #     print(f"[{exp_id}] idx {idx}: clipped {len(dropped)} hallucinated keys {dropped[:5]}")
            # records.append({"idx": idx, "question": ex["question"], "answer": answer,
            #                 "pred": pred, "gt": gt_from_example(ex)})

        except Exception as e:
            print(f"[{exp_id}] idx {idx} FAILED: {type(e).__name__}: {e}")
            records.append({"idx": idx, "error": str(e),
                            "pred": None, "gt": gt_from_example(ex)})
        if len(records) % CKPT_EVERY == 0:
            checkpoint(exp_id, records)

    checkpoint(exp_id, records)
    ok = [r for r in records if r.get("pred")]
    scores = score_experiment(ok)
    print(f"[{exp_id}] {len(ok)}/{len(records)} ok | {scores}")
    return scores

In [16]:
# Cell 16 — SINGLE-EXAMPLE DEBUG GATE (run before any full experiment)
DEBUG_CFG = {"chunking": "sliding", "embedding": "minilm", "retrieval": "hybrid",
             "rerank": True, "ordering": "normal", "generator": "scout", "judge": "70b"}

ex = ds[0]
_emb = get_embedder(DEBUG_CFG["embedding"])
_chunks = build_chunks(ex, DEBUG_CFG["chunking"], _emb)
_idx, _bm = build_indexes(_chunks, _emb, DEBUG_CFG["embedding"])
_ids = hybrid_retrieve(ex["question"], _chunks, _idx, _bm, _emb, DEBUG_CFG["embedding"], k=K*2)
_ids = rerank(ex["question"], _chunks, _ids, k=K)
_ans = generate(ex["question"], _chunks, _ids, DEBUG_CFG["generator"])
_keyed = key_documents(_chunks, _ids)
_j = judge(DEBUG_CFG["judge"], _keyed, ex["question"], _ans)
_pred = compute_trace(_j, _keyed)

print("Q:", ex["question"][:120])
print("A:", _ans[:300])
print("judge fields:", sorted(_j.keys()))
print("rel raw :", _j.get("all_relevant_sentence_keys"))
print("util raw:", _j.get("all_utilized_sentence_keys"))
print("dropped by clipping:", _pred["_dropped_keys"])
print("pred:", {k: v for k, v in _pred.items() if not k.startswith("_")})
print("gt  :", gt_from_example(ex))

assert 0 <= _pred["relevance"] <= 1 and 0 <= _pred["utilization"] <= 1
# sanity: adherent answer with zero utilized keys is contradictory -> investigate
if _pred["adherence"] == 1 and _pred["utilization"] == 0.0:
    print("WARNING: adherence=1 but utilization=0 — check key formats / judge output")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Q: Is ITCAM Agent for WebSphere Applications 7.2.0.0.7 available? Is ITCAM Agent for WebSphere Applications 7.2.0.0.7 avail
A: Yes. 

According to the FIX README ABSTRACT, 7.2.0.0.7 is one of the available versions: 
7.2.0.0.7 [http://www.ibm.com/support/fixcentral/swg/quickorder?product=ibm/Tivoli/Tivoli+Composite+Application+Manager+for+Applications&release=All&platform=All&function=fixId&fixids=7.2.0.0-TIV-ITCAMAD_WS-IF0
judge fields: ['all_relevant_sentence_keys', 'all_utilized_sentence_keys', 'overall_supported', 'overall_supported_explanation', 'relevance_explanation', 'sentence_support_information']
rel raw : ['4a', '4b']
util raw: ['4a', '4b']
dropped by clipping: []
pred: {'relevance': 0.125, 'utilization': 0.125, 'completeness': 1.0, 'adherence': 1}
gt  : {'relevance': 0.017241379310344827, 'utilization': 0.004310344827586207, 'completeness': 0.25, 'adherence': 1}


In [17]:
for eid in ["CS-EXP-016", "CS-EXP-017"]:
    recs = load_checkpoint(eid)
    clean = [r for r in recs if r.get("pred")]
    print(eid, ":", len(recs), "->", len(clean))
    for d in [LOCAL_DIR, DRIVE_DIR]:
        p = os.path.join(d, f"{eid}.json")
        if os.path.exists(p):
            with open(p, "w") as f: json.dump(clean, f, indent=2)
    _last_ckpt_count.pop(eid, None)

[ckpt] resumed CS-EXP-016 at 200 records
CS-EXP-016 : 200 -> 82
[ckpt] resumed CS-EXP-017 at 130 records
CS-EXP-017 : 130 -> 3


## Experiment plan — greedy stage-wise (same as GK)

Fix everything else, sweep one stage, lock the winner, move on.

| Stage | EXPs | Sweep | Fixed |
|---|---|---|---|
| 0 Baseline | EXP-001 | — | fixed + minilm + dense + no rerank + normal + scout, judge=70b |
| 1 Chunking | EXP-002–003 | sliding, semantic | rest = baseline |
| 2 Embedding | EXP-004–005 | bge-large, e5-large | best chunking locked |
| 3 Retrieval | EXP-006–007 | bm25, hybrid RRF | best emb locked |
| 4 Rerank | EXP-008 | +cross-encoder | best retrieval locked |
| 5 Ordering | EXP-009 | reverse | best rerank locked |
| 6 Generator | EXP-010–012 | 8b, 70b, qwen | best ordering locked |
| 7 Judge check | EXP-013 | judge=8b vs 70b on winner | "smaller judge preferred" claim — verify (GK: 8b over-flagged, RMSE halved w/ 70b) |
| Final | EXP-F | winner @ N_SAMPLES=200 | authoritative run |

In [18]:
# Cell 18 — Experiments (run stage by stage; update LOCKED after each stage)
# Cell 18 — FINAL RUN @ N=200 (locked config)
print(f"*** N_SAMPLES = {N_SAMPLES} | SEED = {SEED} ***"); time.sleep(3)

BASE = {"chunking": "sliding", "embedding": "minilm", "retrieval": "dense",
        "rerank": False, "ordering": "normal", "generator": "gpt-oss", "judge": "70b"}

ALL_SCORES = {}

def run(exp_id, **overrides):
    cfg = {**BASE, **overrides}
    ALL_SCORES[exp_id] = {"cfg": cfg, "scores": run_experiment(exp_id, cfg)}

# run("CS-EXP-FINAL")   # done 200/200

# # Stage 0 — baseline (done; resumes from checkpoint, no API calls)
# run("CS-EXP-001")

# # Stage 1 — chunking  (done: sliding wins on adherence AUCROC 0.75)
# run("CS-EXP-002", chunking="sliding")
# run("CS-EXP-003", chunking="semantic")
# BASE["chunking"] = "sliding"   # LOCKED

# # Stage 2 — embedding
# run("CS-EXP-004", embedding="bge-large")
# run("CS-EXP-005", embedding="e5-large")
# BASE["embedding"] = "minilm"   # LOCKED — large embedders no gain (mirrors Biomedical)

# # Stage 3 — retrieval
# run("CS-EXP-006", retrieval="bm25")
# run("CS-EXP-007", retrieval="hybrid")
# BASE["retrieval"] = "dense"   # LOCKED

# # Stage 4 — rerank (direct test of team's hybrid+reranker hypothesis)
# run("CS-EXP-008", rerank=True)                        # dense + reranker
# run("CS-EXP-008b", rerank=True, retrieval="hybrid")   # hybrid + reranker
# BASE["rerank"] = False   # LOCKED — reranking changed context (overlap 0.65) but not adherence

# # Stage 5 — ordering
# run("CS-EXP-009", ordering="reverse")
# BASE["ordering"] = "normal"   # LOCKED — reverse no gain (context too short for LitM effects)

# # Stage 6 — generator  (done: gpt-oss wins 0.875/0.84)
# run("CS-EXP-010", generator="8b")
# run("CS-EXP-011", generator="70b")
# run("CS-EXP-012", generator="gpt-oss")
# BASE["generator"] = "gpt-oss"   # LOCKED — decommission-proof

# # Stage 7 — judge sanity (winner cfg, judge=8b)
# run("CS-EXP-013", judge="8b")


# # Stage 8 — context/prompt strategies (locked config, gpt-oss, 70b judge)
# run("CS-EXP-014", prompt="soft")
# run("CS-EXP-015", compress=True)

# # Transfer validation — does the Stage-1 chunking decision hold under gpt-oss?
# run("CS-VAL-001", chunking="fixed")

# Stage 9 — coverage sweep (root cause: answer-absent at k=5)
# Stage 9 — coverage sweep
run("CS-EXP-016", k=10)          # resumes at ~82, finishing to 200
# run("CS-EXP-017", k=10, rerank=True)
# run("CS-EXP-018", chunking="sliding_L")


*** N_SAMPLES = 200 | SEED = 42 ***
=== CS-EXP-016 | {'chunking': 'sliding', 'embedding': 'minilm', 'retrieval': 'dense', 'rerank': False, 'ordering': 'normal', 'generator': 'gpt-oss', 'judge': '70b', 'k': 10}
[ckpt] resumed CS-EXP-016 at 82 records


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[CS-EXP-016] idx 59: clipped 2 hallucinated keys ['1i', '1k']
[CS-EXP-016] idx 79: clipped 1 hallucinated keys ['6f']
[ckpt] CS-EXP-016: 85 records backed up (local/Drive/GitHub)
[ckpt] CS-EXP-016: 90 records backed up (local/Drive/GitHub)
[ckpt] CS-EXP-016: 95 records backed up (local/Drive/GitHub)
[ckpt] CS-EXP-016: 100 records backed up (local/Drive/GitHub)
[ckpt] CS-EXP-016: 105 records backed up (local/Drive/GitHub)
[ckpt] CS-EXP-016: 110 records backed up (local/Drive/GitHub)
[ckpt] CS-EXP-016: 115 records backed up (local/Drive/GitHub)
[ckpt] CS-EXP-016: 120 records backed up (local/Drive/GitHub)
[ckpt] CS-EXP-016: 125 records backed up (local/Drive/GitHub)
[ckpt] CS-EXP-016: 130 records backed up (local/Drive/GitHub)
[ckpt] CS-EXP-016: 135 records backed up (local/Drive/GitHub)
[ckpt] CS-EXP-016: 140 records backed up (local/Drive/GitHub)
[ckpt] CS-EXP-016: 145 records backed up (local/Drive/GitHub)
[ckpt] CS-EXP-016: 150 records backed up (local/Drive/GitHub)
[ckpt] CS-EXP-016

In [24]:
f = {r["idx"]: r for r in load_checkpoint("CS-EXP-FINAL") if r.get("pred")}
o = {r["idx"]: r for r in load_checkpoint("CS-EXP-012") if r.get("pred")}
common = sorted(set(f) & set(o))
agree = sum(f[i]["pred"]["adherence"] == o[i]["pred"]["adherence"] for i in common)
print(f"adherence agreement on shared 25: {agree}/{len(common)}")

[ckpt] resumed CS-EXP-FINAL at 200 records
[ckpt] resumed CS-EXP-012 at 25 records
adherence agreement on shared 25: 20/25


In [19]:
# Cell 19 — Results table + save
import pandas as pd
rows = []
for eid, v in ALL_SCORES.items():
    r = {"exp": eid, **{k: v["cfg"][k] for k in ["chunking","embedding","retrieval","rerank","ordering","generator","judge"]}, **v["scores"]}
    rows.append(r)
df = pd.DataFrame(rows)
display(df)
df.to_csv(os.path.join(LOCAL_DIR, "cs_summary.csv"), index=False)
df.to_csv(os.path.join(DRIVE_DIR, "cs_summary.csv"), index=False)
_github_push("cs_summary.csv", df.to_csv(index=False))

,exp,chunking,embedding,retrieval,rerank,ordering,generator,judge,relevance_rmse,utilization_rmse,completeness_rmse,adherence_aucroc,adherence_accuracy,idk_rate
0,CS-EXP-016,sliding,minilm,dense,False,normal,gpt-oss,70b,0.325629,0.182594,0.615208,0.6482,0.64,0.32


True

In [20]:
# Build GK-format tracker table for Customer Support
import pandas as pd, numpy as np

CFGS = {  # from ALL_SCORES / the summary table
 "CS-EXP-001": ("Fixed (120w)","all-MiniLM-L6-v2","FAISS (dense)","None","llama-4-scout","Baseline. Acc 0.60."),
 "CS-EXP-002": ("Sliding (120w/40)","all-MiniLM-L6-v2","FAISS (dense)","None","llama-4-scout","Stage-1 winner. Sliding preserves conversational flow. Acc 0.68."),
 "CS-EXP-003": ("Semantic","all-MiniLM-L6-v2","FAISS (dense)","None","llama-4-scout","Best Rel/Comp RMSE, no adherence gain - cleaner sentences but broken dialogue context."),
 "CS-EXP-004": ("Sliding (120w/40)","bge-large-en-v1.5","FAISS (dense)","None","llama-4-scout","Large embedder below minilm."),
 "CS-EXP-005": ("Sliding (120w/40)","e5-large-v2","FAISS (dense)","None","llama-4-scout","No gain vs minilm (delta 0.024 = noise). Mirrors Biomedical finding."),
 "CS-EXP-006": ("Sliding (120w/40)","all-MiniLM-L6-v2","BM25","None","llama-4-scout","Best Rel RMSE (lexical match on error codes) but adherence drops."),
 "CS-EXP-007": ("Sliding (120w/40)","all-MiniLM-L6-v2","Hybrid RRF","None","llama-4-scout","Hybrid below dense - contradicts cross-team note for techqa."),
 "CS-EXP-008": ("Sliding (120w/40)","all-MiniLM-L6-v2","FAISS (dense)","Cross-encoder ms-marco-MiniLM","llama-4-scout","Tied w/ 002 (1 flip, idx24 induced hallucination). Reranker changed 35% of top-5; hallucination not selection-limited."),
 "CS-EXP-008b": ("Sliding (120w/40)","all-MiniLM-L6-v2","Hybrid RRF","Cross-encoder ms-marco-MiniLM","llama-4-scout","Hybrid+reranker (team rec) directly tested - REJECTED, last on adherence."),
 "CS-EXP-009": ("Sliding (120w/40)","all-MiniLM-L6-v2","FAISS (dense)","None","llama-4-scout (reverse ctx)","Reverse ordering (LitM) no gain - k=5 context too short for position effects."),
 "CS-EXP-010": ("Sliding (120w/40)","all-MiniLM-L6-v2","FAISS (dense)","None","llama-3.1-8b-instant","Stage 6. Below scout reference."),
 "CS-EXP-011": ("Sliding (120w/40)","all-MiniLM-L6-v2","FAISS (dense)","None","llama-3.3-70b-versatile","Matches scout (0.750/0.68)."),
 "CS-EXP-012": ("Sliding (120w/40)","all-MiniLM-L6-v2","FAISS (dense)","None","openai/gpt-oss-120b","WINNER + LOCKED. Best adherence recorded. Util RMSE inflated by comprehensive answers vs sparse GT (grounded per adherence). Acc 0.84."),
 "CS-EXP-013": ("Sliding (120w/40)","all-MiniLM-L6-v2","FAISS (dense)","None","openai/gpt-oss-120b (judge=8b)","JUDGE CHECK - NOT comparable to other rows. 8b judge 0.639 vs 70b 0.875 on same pipeline: 'smaller judge preferred' REFUTED; 70b judge mandatory."),
 "CS-EXP-014": ("Sliding (120w/40)","all-MiniLM-L6-v2","FAISS (dense)","None","openai/gpt-oss-120b (soft prompt)","PROMPT TEST - soft/lenient prompt CRASHED adherence (0.600 vs 0.875): strict IDK instruction is protective; leniency trades honest refusals for unsupported partial answers. All RMSEs degraded."),
 "CS-EXP-015": ("Sliding (120w/40)","all-MiniLM-L6-v2","FAISS (dense)","None","openai/gpt-oss-120b (compress-then-answer)","COMPRESSION TEST - no adherence gain (0.694): IDK root cause is retrieval-miss at k=5, not needle-in-noise. Best Comp RMSE (0.49) shows distillation works, but of the wrong haystack."),
 "CS-VAL-001": ("Fixed (120w)","all-MiniLM-L6-v2","FAISS (dense)","None","openai/gpt-oss-120b","TRANSFER VALIDATION - fixed chunking under gpt-oss lands below locked config (0.757 vs 0.875): Stage-1 sliding decision transfers; stage-generator independence empirically supported, not just assumed."),
 "CS-EXP-FINAL": ("Sliding (120w/40)","all-MiniLM-L6-v2","FAISS (dense)","None","openai/gpt-oss-120b","AUTHORITATIVE N=200 FINAL on locked config. AUCROC 0.643 / Acc 0.615 vs 0.875/0.84 @ N=25 - small-sample AUCROC substantially overestimated performance (Finance instability confirmed at scale). 80% per-example agreement w/ EXP-012 on shared 25: drop is sample composition, not run instability. RMSEs improved at scale (util 0.24 vs 0.33). Residual error dominated by answer-absent queries at k=5."),
}

rows = []
for eid, (chunk, emb, ret, rr, gen, note) in CFGS.items():
    recs = [r for r in load_checkpoint(eid) if r.get("pred")]
    n = len(recs)
    def mp(m): return np.mean([r["pred"][m] for r in recs])
    def rmse(m):
        pairs = [(r["pred"][m], r["gt"][m]) for r in recs if r["gt"][m] is not None]
        return math.sqrt(np.mean([(p-g)**2 for p, g in pairs]))
    def refm(m):
        vals = [r["gt"][m] for r in recs if r["gt"][m] is not None]
        return np.mean(vals)
    adh_gt = [r["gt"]["adherence"] for r in recs if r["gt"]["adherence"] is not None]
    adh_pr = [r["pred"]["adherence"] for r in recs if r["gt"]["adherence"] is not None]
    auc = roc_auc_score(adh_gt, adh_pr) if len(set(adh_gt)) > 1 else None
    rows.append({
        "Experiment ID": eid, "Domain": "Customer Support (techqa)",
        "Chunking Strategy": chunk, "Embedding Model": emb,
        "Retrieval Technique": ret, "Re-ranking": rr, "Generator LLM Used": gen,
        "Context Relevance (score) / Rel RMSE": f"{mp('relevance'):.4f}\nRMSE {rmse('relevance'):.4f}",
        "Context Utilization (score) / Util RMSE": f"{mp('utilization'):.4f}\nRMSE {rmse('utilization'):.4f}",
        "Completeness (score) / Comp RMSE": f"{mp('completeness'):.4f}\nRMSE {rmse('completeness'):.4f}",
        "Adherence (score) / Adh AUCROC": f"{np.mean(adh_pr):.4f}\nAUCROC {auc:.4f}" if auc else f"{np.mean(adh_pr):.4f}\nAUCROC n/a",
        "Reference Relevance": f"{refm('relevance'):.4f}",
        "Reference Utilization": f"{refm('utilization'):.4f}",
        "Reference Completeness": f"{refm('completeness'):.4f}",
        "Reference Adherence": f"{refm('adherence'):.4f}",
        "Samples": n, "Notes / Error Analysis": note,
    })

tracker = pd.DataFrame(rows)
display(tracker)
tracker.to_csv(os.path.join(LOCAL_DIR, "cs_tracker_gk_format.csv"), index=False)
tracker.to_csv(os.path.join(DRIVE_DIR, "cs_tracker_gk_format.csv"), index=False)
_github_push("cs_tracker_gk_format.csv", tracker.to_csv(index=False))
print("saved: cs_tracker_gk_format.csv")

[ckpt] resumed CS-EXP-001 at 25 records
[ckpt] resumed CS-EXP-002 at 25 records
[ckpt] resumed CS-EXP-003 at 25 records
[ckpt] resumed CS-EXP-004 at 25 records
[ckpt] resumed CS-EXP-005 at 25 records
[ckpt] resumed CS-EXP-006 at 25 records
[ckpt] resumed CS-EXP-007 at 25 records
[ckpt] resumed CS-EXP-008 at 25 records
[ckpt] resumed CS-EXP-008b at 25 records
[ckpt] resumed CS-EXP-009 at 25 records
[ckpt] resumed CS-EXP-010 at 25 records
[ckpt] resumed CS-EXP-011 at 25 records
[ckpt] resumed CS-EXP-012 at 25 records
[ckpt] resumed CS-EXP-013 at 25 records
[ckpt] resumed CS-EXP-014 at 25 records
[ckpt] resumed CS-EXP-015 at 25 records
[ckpt] resumed CS-VAL-001 at 25 records
[ckpt] resumed CS-EXP-FINAL at 200 records


,Experiment ID,Domain,Chunking Strategy,Embedding Model,Retrieval Technique,Re-ranking,Generator LLM Used,Context Relevance (score) / Rel RMSE,Context Utilization (score) / Util RMSE,Completeness (score) / Comp RMSE,Adherence (score) / Adh AUCROC,Reference Relevance,Reference Utilization,Reference Completeness,Reference Adherence,Samples,Notes / Error Analysis
0,CS-EXP-001,Customer Support (techqa),Fixed (120w),all-MiniLM-L6-v2,FAISS (dense),None,llama-4-scout,0.4110\nRMSE 0.4401,0.0757\nRMSE 0.0918,0.1799\nRMSE 0.6226,0.4000\nAUCROC 0.6389,0.0750,0.0435,0.5394,0.6400,25,Baseline. Acc 0.60.
1,CS-EXP-002,Customer Support (techqa),Sliding (120w/40),all-MiniLM-L6-v2,FAISS (dense),None,llama-4-scout,0.4654\nRMSE 0.4696,0.1220\nRMSE 0.1813,0.2141\nRMSE 0.5892,0.3200\nAUCROC 0.7500,0.0750,0.0435,0.5394,0.6400,25,Stage-1 winner. Sliding preserves conversation...
2,CS-EXP-003,Customer Support (techqa),Semantic,all-MiniLM-L6-v2,FAISS (dense),None,llama-4-scout,0.3655\nRMSE 0.3846,0.1163\nRMSE 0.1551,0.3104\nRMSE 0.5468,0.3200\nAUCROC 0.6632,0.0750,0.0435,0.5394,0.6400,25,"Best Rel/Comp RMSE, no adherence gain - cleane..."
3,CS-EXP-004,Customer Support (techqa),Sliding (120w/40),bge-large-en-v1.5,FAISS (dense),None,llama-4-scout,0.4562\nRMSE 0.4815,0.1047\nRMSE 0.1468,0.2076\nRMSE 0.5855,0.3600\nAUCROC 0.6944,0.0750,0.0435,0.5394,0.6400,25,Large embedder below minilm.
4,CS-EXP-005,Customer Support (techqa),Sliding (120w/40),e5-large-v2,FAISS (dense),None,llama-4-scout,0.4845\nRMSE 0.4980,0.0739\nRMSE 0.0869,0.1432\nRMSE 0.6194,0.4000\nAUCROC 0.7257,0.0750,0.0435,0.5394,0.6400,25,No gain vs minilm (delta 0.024 = noise). Mirro...
5,CS-EXP-006,Customer Support (techqa),Sliding (120w/40),all-MiniLM-L6-v2,BM25,None,llama-4-scout,0.3510\nRMSE 0.3752,0.1005\nRMSE 0.1532,0.2829\nRMSE 0.5871,0.3200\nAUCROC 0.6632,0.0750,0.0435,0.5394,0.6400,25,Best Rel RMSE (lexical match on error codes) b...
6,CS-EXP-007,Customer Support (techqa),Sliding (120w/40),all-MiniLM-L6-v2,Hybrid RRF,None,llama-4-scout,0.4367\nRMSE 0.4471,0.0989\nRMSE 0.1391,0.2391\nRMSE 0.5873,0.3600\nAUCROC 0.6944,0.0750,0.0435,0.5394,0.6400,25,Hybrid below dense - contradicts cross-team no...
7,CS-EXP-008,Customer Support (techqa),Sliding (120w/40),all-MiniLM-L6-v2,FAISS (dense),Cross-encoder ms-marco-MiniLM,llama-4-scout,0.4535\nRMSE 0.4685,0.0672\nRMSE 0.0821,0.1737\nRMSE 0.6185,0.4000\nAUCROC 0.7257,0.0750,0.0435,0.5394,0.6400,25,"Tied w/ 002 (1 flip, idx24 induced hallucinati..."
8,CS-EXP-008b,Customer Support (techqa),Sliding (120w/40),all-MiniLM-L6-v2,Hybrid RRF,Cross-encoder ms-marco-MiniLM,llama-4-scout,0.3922\nRMSE 0.3902,0.0772\nRMSE 0.0892,0.1917\nRMSE 0.6046,0.3200\nAUCROC 0.6632,0.0750,0.0435,0.5394,0.6400,25,Hybrid+reranker (team rec) directly tested - R...
9,CS-EXP-009,Customer Support (techqa),Sliding (120w/40),all-MiniLM-L6-v2,FAISS (dense),None,llama-4-scout (reverse ctx),0.5052\nRMSE 0.5098,0.0895\nRMSE 0.1279,0.1582\nRMSE 0.6463,0.2800\nAUCROC 0.7188,0.0750,0.0435,0.5394,0.6400,25,Reverse ordering (LitM) no gain - k=5 context ...


saved: cs_tracker_gk_format.csv
